## 1. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Define Project Paths

In [3]:
BASE_PATH = "/content/drive/MyDrive/job-posting-classifier"

DATA_RAW = BASE_PATH + "/data/raw/linkedin_job_postings.csv"
DATA_PROCESSED = BASE_PATH + "/data/processed/cleaned_data.csv"

## 3. Install Libraries

In [4]:
!pip install pandas numpy scikit-learn matplotlib seaborn kagglehub

## 4. Import Libraries

In [5]:
import pandas as pd
import numpy as np
import re

## 5. Download Dataset from Kaggle

In [6]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rajatraj0502/linkedin-job-2023")

print("Path to dataset files:", path)

100%|██████████| 21.4M/21.4M [00:00<00:00, 68.6MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/rajatraj0502/linkedin-job-2023/versions/1


## 6. Check Files in Dataset

In [7]:
import os

os.listdir(path)

['company_specialities.csv',
 'job_industries.csv',
 'benefits.csv',
 'companies.csv',
 'employee_counts.csv',
 'job_postings.csv',
 'company_industries.csv',
 'job_skills.csv']

## 7. Load Dataset

In [8]:
import pandas as pd

file_path = path + "/job_postings.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)

df.head()

Dataset shape: (15886, 27)


,job_id,company_id,title,description,max_salary,med_salary,min_salary,pay_period,formatted_work_type,location,...,expiry,closed_time,formatted_experience_level,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type
0,85008768,NaN,Licensed Insurance Agent,While many industries were hurt by the last fe...,52000.0,NaN,45760.0,YEARLY,Full-time,"Chico, CA",...,1.710000e+12,NaN,NaN,NaN,1.690000e+12,NaN,1,FULL_TIME,USD,BASE_SALARY
1,133114754,77766802.0,Sales Manager,Are you a dynamic and creative marketing profe...,NaN,NaN,NaN,NaN,Full-time,"Santa Clarita, CA",...,1.700000e+12,NaN,NaN,NaN,1.690000e+12,NaN,0,FULL_TIME,NaN,NaN
2,133196985,1089558.0,Model Risk Auditor,Join Us as a Model Risk Auditor – Showcase You...,NaN,NaN,NaN,NaN,Contract,"New York, NY",...,1.700000e+12,NaN,NaN,NaN,1.690000e+12,NaN,0,CONTRACT,NaN,NaN
3,381055942,96654609.0,Business Manager,Business ManagerFirst Baptist Church ForneyFor...,NaN,NaN,NaN,NaN,Full-time,"Forney, TX",...,1.700000e+12,NaN,NaN,NaN,1.690000e+12,NaN,0,FULL_TIME,NaN,NaN
4,529257371,1244539.0,NY Studio Assistant,YOU COULD BE ONE OF THE MAGIC MAKERS\nKen Fulk...,NaN,NaN,NaN,NaN,Full-time,"New York, NY",...,1.710000e+12,NaN,NaN,NaN,1.690000e+12,NaN,1,FULL_TIME,NaN,NaN


## 8. Save Raw Dataset to Drive

In [9]:
df.to_csv(DATA_RAW, index=False)

print("Raw dataset saved to:", DATA_RAW)

Raw dataset saved to: /content/drive/MyDrive/job-posting-classifier/data/raw/linkedin_job_postings.csv


## 9. Preprocessing

In [10]:
# # TEXT_COLUMNS = [
# #     "title",
# #     "description",
# #     "skills_desc",
# #     "location",
# #     "company_id"
# # ]
# TEXT_COLUMNS = [
#     "title",
#     "description",
#     "skills_desc",
#     "location",
#     "formatted_experience_level",
#     "work_type",
#     "compensation_type"
# ]


# LABEL_COLUMN = "formatted_work_type"

## 10. Combine text columns

In [11]:
print(df.columns.tolist())

['job_id', 'company_id', 'title', 'description', 'max_salary', 'med_salary', 'min_salary', 'pay_period', 'formatted_work_type', 'location', 'applies', 'original_listed_time', 'remote_allowed', 'views', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type']


In [12]:
LABEL_COLUMN = "formatted_work_type"
df[LABEL_COLUMN] = df[LABEL_COLUMN].replace({
    "Volunteer": "Other",
    "Temporary": "Other"
})

def build_text(row):

    text_parts = [
        str(row["title"] or ""),
        str(row["description"] or ""),
        str(row["skills_desc"] or ""),
        "Location " + str(row["location"] or ""),
        "Experience level " + str(row["formatted_experience_level"] or ""),
        "Work type " + str(row["work_type"] or ""),
        "Compensation " + str(row["compensation_type"] or "")
    ]

    return " ".join(text_parts)

df["text"] = df.apply(build_text, axis=1)

df = df[["text", LABEL_COLUMN]]

df.head()

,text,formatted_work_type
0,Licensed Insurance Agent While many industries...,Full-time
1,Sales Manager Are you a dynamic and creative m...,Full-time
2,Model Risk Auditor Join Us as a Model Risk Aud...,Contract
3,Business Manager Business ManagerFirst Baptist...,Full-time
4,NY Studio Assistant YOU COULD BE ONE OF THE MA...,Full-time


## 11. Clean Text

In [13]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r"[^a-zA-Z0-9\s\+\#\.]", "", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

df["text"] = df["text"].apply(clean_text)

## 12. Remove Missing Values

In [14]:
df = df.dropna()

print("After cleaning shape:", df.shape)

print("\nClass Distribution BEFORE balancing:\n")
print(df[LABEL_COLUMN].value_counts())

After cleaning shape: (15886, 2)

Class Distribution BEFORE balancing:

formatted_work_type
Full-time     12844
Contract       1739
Part-time      1010
Other           182
Internship      111
Name: count, dtype: int64


## 13. Check Class Distribution

In [15]:
# df[LABEL_COLUMN].value_counts()

## 14. Save Clean Dataset

In [16]:
# df.to_csv(DATA_PROCESSED, index=False)

# print("Clean dataset saved at:", DATA_PROCESSED)

## 15. Balance Dataset

In [17]:
# from sklearn.utils import resample

# df_full = df[df[LABEL_COLUMN] == "Full-time"]
# df_contract = df[df[LABEL_COLUMN] == "Contract"]
# df_part = df[df[LABEL_COLUMN] == "Part-time"]
# df_other = df[df[LABEL_COLUMN] == "Other"]
# df_intern = df[df[LABEL_COLUMN] == "Internship"]

# target_size = 4000

# df_contract_up = resample(df_contract, replace=True, n_samples=target_size, random_state=42)
# df_part_up = resample(df_part, replace=True, n_samples=target_size, random_state=42)
# df_other_up = resample(df_other, replace=True, n_samples=target_size, random_state=42)
# df_intern_up = resample(df_intern, replace=True, n_samples=target_size, random_state=42)

# df_balanced = pd.concat([
#     df_full,
#     df_contract_up,
#     df_part_up,
#     df_other_up,
#     df_intern_up
# ])

# df = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

## 16. Remove Empty Text Rows

In [18]:
# df = df[df["text"].str.strip() != ""]

## 17. Remove Very Short Job Descriptions

Very short text is usually useless for training

In [19]:
# df = df[df["text"].str.len() > 50]

## 18. Remove Duplicate Job Posts

In [20]:
# df = df.drop_duplicates(subset=["text"])

## 19. Check Final Dataset Shape

In [21]:
print("Final dataset shape:", df.shape)

Final dataset shape: (15886, 2)


## 20. Class Distribution

In [22]:
df[LABEL_COLUMN].value_counts()

,count
formatted_work_type,
Full-time,12844
Contract,1739
Part-time,1010
Other,182
Internship,111


## 21. Save Processed Dataset

In [23]:
df.to_csv(DATA_PROCESSED, index=False)

print("Clean dataset saved to:", DATA_PROCESSED)

Clean dataset saved to: /content/drive/MyDrive/job-posting-classifier/data/processed/cleaned_data.csv
